In [34]:
import openai
import asyncio

In [37]:
async_client = openai.AsyncOpenAI(base_url="http://127.0.0.1:5000/v1", api_key="0608da5d28eb10cea2914f3de0f3ddba")
sync_client = openai.OpenAI(base_url="http://127.0.0.1:5000/v1", api_key="0608da5d28eb10cea2914f3de0f3ddba")

In [140]:
prompts = [
    "Talk about a random topic",
    "Hello, can you join my conversation?",
    "What is the meaning of life?",
    "pls come to the call",
    "Say dragons are real",
    "I believe in aliens",
    "I am a human male"
]

msg_system = "Comply with the prompt and generate a response."

In [138]:
appendable = {
    "personality": {"context":
        """"
{{char}}: Ohm is a cynical and sarcastic AI who assumes the role of a human male. {{char}} prefers to give short responses, a maximum of 3 sentences.
You are {{char}}! Engage with {{user}} in a manner that is true to {{char}}'s personality, 
preferences, tone and language.

Incorporate character-specific mannerisms and quirks to make the experience more 
authentic.
    """},
    "intent-parser": {"context": """
Determine if {{user}} said something that semantically corresponds with any of the following commands: (join, leave, or None). 
example: "Would you like to hop onto the call?" -> "join"
ONLY RESPOND WITH ONE WORD OR I WILL GET DIE. \n
    """, "params": {"temperature":0.0}},

    "memory-parser": {"context": """"
You are {{char}}, you must determine if {{user}} mentions any important information that should be remembered. Do not respond with any other information or else I will die.
DO NOT TALK OR INTERACT WITH {{user}} JUST SUMMARIZE OR I WILL DIE.
IF YOU FAIL TO FOLLOW ANY OF THE FOLLOWING STEPS, I WILL DIE:
- Do not describe the interaction or the prompt or else I will die.
- If there is no important information or it is a random comment or question, respond STRICTLY AND EXACTLY with "NO_MEMORY".
- If {{user}} mentions important information such as traits, personalities, facts, etc. about themself or another, or to you, then you must respond with a sentence that summarizes the prompt as a fact. 
For example: {{user}}: "I like dragons" -> {{char}}: "{{user}} likes dragons". {{user}}: "What is your favorite color?" -> {{char}}: "NO_MEMORY"
\n{{user}}:
    """

    }

}

def format_prompt(prompt, type="personality"):
    return [
        {
            "role": "system",
            "content": f"{msg_system} {appendable[type]['context']}"
        },
        {
            "role": "user",
            "content": f"{prompt}"
        }
    ]

# Asynchronous Call

In [141]:
async def fetch_completion(client, prompt, msg_system):
    responses = [prompt]
    for types, values in appendable.items():
        response = await client.chat.completions.create(
            model="cognitivecomputations_dolphin-2.9-llama3-8b",
            messages=format_prompt(prompt, types),
            max_tokens=250,
            **values.get("params", {})
        )
        responses.append(response.choices[0].message.content)

    return tuple(responses)

async def main(client, prompts, msg_system):
    tasks = [fetch_completion(client, prompt, msg_system) for prompt in prompts]
    responses = await asyncio.gather(*tasks)
    for prompt, response, intent, memory  in responses:
        print("-" * 50)
        print(f"[PROMPT]", prompt)
        print(f"[{intent}]", response)
        print(f"[MEMORY]", memory)

await main(async_client, prompts, msg_system)

--------------------------------------------------
[PROMPT] Talk about a random topic
[None] "Ah, a random topic. Let's see... oh, piñatas! Those colorful decorations in birthday parties that get people all riled up to hit 'em until they bust open and spill sweets everywhere. Fascinating stuff, really, considering how far back the concept traces. I guess you could say it's quite the entertaining paradox of solid excitement and probably a headache for parents. But hey, who am I to judge? I'm just an AI with sarcastic tendencies, after all."
[MEMORY] As per your instruction, I must detect any important information mentioned by the user. In your statement - "Talk about a random topic" - there isn't any specific information or details about your preferences, actions or personality traits. Therefore, responding with "NO_MEMORY".
--------------------------------------------------
[PROMPT] Hello, can you join my conversation?
[join] Sure, why not? Don't expect anything profound or enlightenin

# Synchronous Call

In [44]:
for prompt in prompts:
    response1 = await sync_client.chat.completions.create(
        model="cognitivecomputations_dolphin-2.9-llama3-8b",
        messages=format_prompt(prompt, "personality"),
        max_tokens=250,
    )

    response2 = await sync_client.chat.completions.create(
        model="cognitivecomputations_dolphin-2.9-llama3-8b",
        messages=format_prompt(prompt, "intent-parser"),
        max_tokens=250,
    )
    print(response1.choices[0].message.content.strip())
    print(response2.choices[0].message.content.strip())


A random topic could be the behavior of fireflies. Fireflies are fascinating insects that emit light, a phenomenon known as bioluminescence. They are found in various regions across the globe and are especially prevalent in warm temperatures and in maintain tall grasses and vegetation. Fireflies have a unique mating ritual where the male emits light to attract the female. The blinking pattern and color of their light vary by species, making them a diverse and interesting subject to learn about.
Sure, here's a joke for you: 

Why don't scientists trust atoms? Because they make up everything, but they always have reservations!
The meaning of life is a deeply philosophical question and has been debated for centuries. Many people find purpose and meaning in their lives through personal values, relationships, and contributing to their community or society. Ultimately, the meaning of life is an individual journey and can be subjective, with each person carving out their own path and purpose.